In [1]:
import numpy as np
from scipy.stats import norm
from scipy.fft import fft, ifft

In [2]:
#IEOR 4732 - Case Study 1
#UNI: pp2955
"""
Black-Scholes Option Pricing via:
  (a) Fast Fourier Transform
  (b) Fractional Fast Fourier Transform
  (c) Fourier-cosine Method
"""

'\nBlack-Scholes Option Pricing via:\n  (a) Fast Fourier Transform\n  (b) Fractional Fast Fourier Transform\n  (c) Fourier-cosine Method\n'

In [3]:
#Given params
S0 = 1900.0 #spot price
T = 0.25 #maturity
sigma = 0.36 #volatility
r = 0.02 #risk-free interest rate
q = 0.0187 #continuous dividend rate
strikes = [2000, 2100, 2200] #strike range for european call options

In [4]:
#Benchmark Black-Scholes prices
def bs_call(S, K, T, r, q, sigma):
    d1 = (np.log(S/K)+(r-q+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1-sigma*np.sqrt(T)
    return S*np.exp(-q*T)*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)

bs_prices = {K: bs_call(S0, K, T, r, q, sigma) for K in strikes}
print("Black-Scholes Analytical Prices")
for K, v in bs_prices.items():
    print(f"  K={K}: {v:.6f}")
print()

Black-Scholes Analytical Prices
  K=2000: 95.246692
  K=2100: 64.834620
  K=2200: 42.947175



In [5]:
#Characteristic function of the log of stock price in the Black-Scholes framework
def cf_bs(u, S0, r, q, sigma, T):
    s0 = np.log(S0)
    mu = s0 + (r-q-0.5*sigma**2)*T
    return np.exp(1j*u*mu-0.5*sigma**2*u**2*T)

In [6]:
#(a) Fast Fourier Transform
def fft_call_prices(S0, r, q, sigma, T, eta, alpha, N):
    lam = 2*np.pi/(N*eta)  
    beta = np.log(S0)-lam*N/2

    j = np.arange(1, N+1)
    nu_j = (j-1)*eta

    # Simpson weights for trapezoidal approximation with correction at j=1
    w = np.ones(N)
    w[0] = 0.5

    phi = cf_bs(nu_j-(alpha+1)*1j, S0, r, q, sigma, T)
    denom = (alpha+1j*nu_j)*(alpha+1j*nu_j+1)
    x = eta*(np.exp(-r*T)/denom)*np.exp(-1j*beta*nu_j)*phi*w

    # FFT
    y = fft(x)

    k_m = beta+(j-1)*lam
    C = np.real(y)*np.exp(-alpha*k_m)/np.pi
    return k_m, C

In [7]:
#(a) Fast Fourier Transform Results
eta = 0.25
alphas = [0.4, 1.0, 1.4, 3.0]
ns = [9, 11, 13, 15]

fft_results = {}
for n in ns:
    N = 2**n
    lam = 2*np.pi/(N*eta)
    for alpha in alphas:
        k_grid, prices = fft_call_prices(S0, r, q, sigma, T, eta, alpha, N)
        row = {}
        for K in strikes:
            lnK = np.log(K)
            row[K] = np.interp(lnK, k_grid, prices)
        fft_results[(n, alpha)] = row
        errs = {K: row[K] - bs_prices[K] for K in strikes}
        print(f"n={n:2d} (N={N:5d}), alpha={alpha:.1f}: "
              f"K=2000:{row[2000]:.4f}(err={errs[2000]:+.4f})  "
              f"K=2100:{row[2100]:.4f}(err={errs[2100]:+.4f})  "
              f"K=2200:{row[2200]:.4f}(err={errs[2200]:+.4f})")
    print()

n= 9 (N=  512), alpha=0.4: K=2000:95.5025(err=+0.2558)  K=2100:65.0559(err=+0.2213)  K=2200:43.0761(err=+0.1289)
n= 9 (N=  512), alpha=1.0: K=2000:95.4211(err=+0.1744)  K=2100:64.9745(err=+0.1399)  K=2200:42.9946(err=+0.0475)
n= 9 (N=  512), alpha=1.4: K=2000:95.4211(err=+0.1744)  K=2100:64.9745(err=+0.1399)  K=2200:42.9946(err=+0.0475)
n= 9 (N=  512), alpha=3.0: K=2000:95.4211(err=+0.1744)  K=2100:64.9745(err=+0.1399)  K=2200:42.9946(err=+0.0475)

n=11 (N= 2048), alpha=0.4: K=2000:95.3659(err=+0.1192)  K=2100:64.9477(err=+0.1131)  K=2200:43.0395(err=+0.0924)
n=11 (N= 2048), alpha=1.0: K=2000:95.2845(err=+0.0378)  K=2100:64.8663(err=+0.0317)  K=2200:42.9581(err=+0.0110)
n=11 (N= 2048), alpha=1.4: K=2000:95.2845(err=+0.0378)  K=2100:64.8663(err=+0.0317)  K=2200:42.9581(err=+0.0110)
n=11 (N= 2048), alpha=3.0: K=2000:95.2845(err=+0.0378)  K=2100:64.8663(err=+0.0317)  K=2200:42.9581(err=+0.0110)

n=13 (N= 8192), alpha=0.4: K=2000:95.3314(err=+0.0847)  K=2100:64.9196(err=+0.0850)  K=2200:43

In [8]:
#(b) Fractional Fast Fourier Transform
def frfft_call_prices(S0, r, q, sigma, T, eta, lam, alpha, N):
    gamma = eta*lam/(2*np.pi)
    beta = np.log(S0)-lam*N/2

    j = np.arange(1, N+1)
    nu_j = (j-1)*eta

    w = np.ones(N); w[0] = 0.5
    denom = (alpha+1j*nu_j)*(alpha+1j*nu_j+1)
    phi = cf_bs(nu_j-(alpha+1)*1j, S0, r, q, sigma, T)
    x = eta*(np.exp(-r*T)/denom)*np.exp(-1j*beta*nu_j)*phi*w

    # Build y and z (2N-long)
    y = np.zeros(2*N, dtype=complex)
    z = np.zeros(2*N, dtype=complex)

    idx = np.arange(N)
    y[:N] = x*np.exp(-1j*np.pi*idx**2*gamma)

    z[:N] = np.exp(1j*np.pi*idx**2*gamma)
    idx2 = np.arange(N, 2*N)
    z[N:] = np.exp(1j*np.pi*(2*N-idx2)**2*gamma)

    # Three FFT steps
    Y_hat = fft(y)
    Z_hat = fft(z)
    xi = Y_hat * Z_hat
    xi_hat = ifft(xi)   # IFFT

    # Take first N elements, apply phase correction
    m_idx = np.arange(N)
    phase = np.exp(-1j*np.pi*m_idx**2*gamma)
    G = xi_hat[:N]*phase

    # Log-strike grid
    k_m = beta+m_idx*lam

    # Call prices
    exp_factor = np.exp(-alpha*k_m)
    C = (exp_factor/np.pi)*np.real(G)
    return k_m, C

In [9]:
#(b) Fractional Fast Fourier Transform Results
eta_fr = 0.25
lam_fr = 0.1
ns_fr  = [6, 7, 8, 9]

frfft_results = {}
for n in ns_fr:
    N = 2**n
    for alpha in alphas:
        k_grid, prices = frfft_call_prices(S0, r, q, sigma, T, eta_fr, lam_fr, alpha, N)
        row = {}
        for K in strikes:
            lnK = np.log(K)
            row[K] = np.interp(lnK, k_grid, prices)
        frfft_results[(n, alpha)] = row
        errs = {K: row[K] - bs_prices[K] for K in strikes}
        print(f"n={n}, (N={N:3d}), alpha={alpha:.1f}: "
              f"K=2000:{row[2000]:.4f}(err={errs[2000]:+.4f})  "
              f"K=2100:{row[2100]:.4f}(err={errs[2100]:+.4f})  "
              f"K=2200:{row[2200]:.4f}(err={errs[2200]:+.4f})")
    print()

n=6, (N= 64), alpha=0.4: K=2000:99.5917(err=+4.3450)  K=2100:64.9534(err=+0.1188)  K=2200:46.4923(err=+3.5452)
n=6, (N= 64), alpha=1.0: K=2000:99.5105(err=+4.2638)  K=2100:64.8885(err=+0.0539)  K=2200:46.4263(err=+3.4792)
n=6, (N= 64), alpha=1.4: K=2000:99.5069(err=+4.2602)  K=2100:64.8955(err=+0.0609)  K=2200:46.4354(err=+3.4882)
n=6, (N= 64), alpha=3.0: K=2000:99.4707(err=+4.2240)  K=2100:64.8881(err=+0.0535)  K=2200:46.4515(err=+3.5043)

n=7, (N=128), alpha=0.4: K=2000:99.5549(err=+4.3083)  K=2100:64.9283(err=+0.0937)  K=2200:46.5028(err=+3.5556)
n=7, (N=128), alpha=1.0: K=2000:99.4735(err=+4.2268)  K=2100:64.8469(err=+0.0122)  K=2200:46.4214(err=+3.4742)
n=7, (N=128), alpha=1.4: K=2000:99.4735(err=+4.2268)  K=2100:64.8469(err=+0.0122)  K=2200:46.4214(err=+3.4742)
n=7, (N=128), alpha=3.0: K=2000:99.4735(err=+4.2268)  K=2100:64.8469(err=+0.0122)  K=2200:46.4214(err=+3.4742)

n=8, (N=256), alpha=0.4: K=2000:99.5549(err=+4.3083)  K=2100:64.9283(err=+0.0937)  K=2200:46.5028(err=+3.5556)

In [10]:
#(c) Fourier-cosine Method
def chi_k(k, a, b, c, d):
    if k == 0:
        return np.exp(d)-np.exp(c)
    term = k*np.pi/(b-a)
    denom = 1+term**2
    val = (np.cos(term*(d-a))*np.exp(d)-np.cos(term*(c-a))*np.exp(c)
           +term*np.sin(term*(d-a))*np.exp(d)-term*np.sin(term*(c-a))*np.exp(c))/denom
    return val

In [11]:
def psi_k(k, a, b, c, d):
    """Integral of cos(k*pi*(y-a)/(b-a)) from c to d"""
    if k == 0:
        return d-c
    term = k*np.pi/(b-a)
    return (np.sin(term*(d-a))-np.sin(term*(c-a)))/term

In [12]:
def cos_call_price(S0, K, r, q, sigma, T, a, b, N_cos):
    x = np.log(S0/K)

    k_arr = np.arange(N_cos)
    u_k = k_arr*np.pi/(b-a)

    phi_k = cf_bs(u_k, S0, r, q, sigma, T) * np.exp(-1j*u_k*np.log(K))

    F_k = np.real(phi_k*np.exp(-1j*k_arr*np.pi*a/(b-a)))

    chi = np.array([chi_k(k, a, b, 0.0, b) for k in k_arr])
    psi = np.array([psi_k(k, a, b, 0.0, b) for k in k_arr])
    V_k = (2/(b-a))*K*(chi-psi)

    # Weighted sum (first term halved)
    F_k[0] *= 0.5

    price = np.exp(-r*T)*np.dot(F_k, V_k)
    return price

In [13]:
#(c) Fourier-cosine Method Results
N_cos     = 200
intervals = [(-1, 1), (-4, 4), (-8, 8), (-12, 12)]
cos_results = {}
for (a, b) in intervals:
    row = {}
    for K in strikes:
        price = cos_call_price(S0, K, r, q, sigma, T, a, b, N_cos)
        row[K] = price
    cos_results[(a, b)] = row
    errs = {K: row[K] - bs_prices[K] for K in strikes}
    # print(cos_results)
    print(f"  [a,b]=[{a:3d},{b:3d}]: "
          f"K=2000:{row[2000]:.6f}(err={errs[2000]:+.6f})  "
          f"K=2100:{row[2100]:.6f}(err={errs[2100]:+.6f})  "
          f"K=2200:{row[2200]:.6f}(err={errs[2200]:+.6f})")


  [a,b]=[ -1,  1]: K=2000:95.246692(err=-0.000000)  K=2100:64.834620(err=-0.000000)  K=2200:42.947175(err=-0.000000)
  [a,b]=[ -4,  4]: K=2000:95.246692(err=-0.000000)  K=2100:64.834620(err=-0.000000)  K=2200:42.947175(err=-0.000000)
  [a,b]=[ -8,  8]: K=2000:95.246692(err=+0.000000)  K=2100:64.834620(err=-0.000000)  K=2200:42.947175(err=-0.000000)
  [a,b]=[-12, 12]: K=2000:95.594927(err=+0.348235)  K=2100:65.212643(err=+0.378023)  K=2200:42.824468(err=-0.122708)
